<a href="https://colab.research.google.com/github/bidallei/MIAAD-UACJ/blob/main/SistemaRec_Conocimiento_AlvaroHernandez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica: Sistema de recomendación basado en conocimiento mediante reglas 2

* Asignatura: Sistemas de recomendación
* Profesor: Dra. Karla Olmos Sánchez
* Alumno: **Álvaro Hernández Jarquín**
* Matrícula: 263150
* Fecha de entrega: 20 de octubre de 2026




In [1]:
SERVICIOS = [
    dict(id='S1', nombre='Consulta medicina familiar - Centro A', tipo='presencial',
         especialidad='general', tiempo_espera_dias=1, requiere_derivacion=False,
         tiene_equipo_estudios=False, edad_minima=18, zona='Norte', centro='Centro A'),

    dict(id='S2', nombre='Ginecología - Hospital B', tipo='presencial',
         especialidad='especialidad', tiempo_espera_dias=5, requiere_derivacion=True,
         tiene_equipo_estudios=True, edad_minima=18, zona='Sur', centro='Hospital B'),

    dict(id='S3', nombre='Mastografía preventiva - Clínica C', tipo='estudio',
         especialidad='imagenologia', tiempo_espera_dias=2, requiere_derivacion=False,
         tiene_equipo_estudios=True, edad_minima=40, zona='Norte', centro='Clínica C'),

    dict(id='S4', nombre='Orientación médica virtual', tipo='virtual',
         especialidad='general', tiempo_espera_dias=0, requiere_derivacion=False,
         tiene_equipo_estudios=False, edad_minima=18, zona='Todas', centro='Digital'),

    dict(id='S5', nombre='Urgencias ginecológicas - Hospital B', tipo='presencial',
         especialidad='especialidad', tiempo_espera_dias=0, requiere_derivacion=False,
         tiene_equipo_estudios=True, edad_minima=18, zona='Sur', centro='Hospital B')
]

In [2]:
PACIENTE_1 = dict(
    nombre='Pancracia', edad=32, urgencia_dias=1, tiene_derivacion=False,
    requiere_equipo=False, zona_residencia='Norte', centro_previo='Centro A',
    antecedentes_riesgo=False, dolor_intensidad=8
)

PACIENTE_2 = dict(
    nombre='Ricarda', edad=45, urgencia_dias=15, tiene_derivacion=True,
    requiere_equipo=True, zona_residencia='Sur', centro_previo='Hospital B',
    antecedentes_riesgo=True, dolor_intensidad=2
)

In [3]:
def evaluar_restricciones_salud(servicio, paciente):
    razones = []
    # R1: Urgencia vs Tiempo de espera
    if servicio['tiempo_espera_dias'] > paciente['urgencia_dias']:
        razones.append('R1: Tiempo de espera supera la tolerancia clínica')
    # R2: Requisito de derivación
    if servicio['requiere_derivacion'] and not paciente['tiene_derivacion']:
        razones.append('R2: Requiere pase a especialidad')
    # R3: Capacidad técnica
    if paciente['requiere_equipo'] and not servicio['tiene_equipo_estudios']:
        razones.append('R3: El centro no cuenta con el equipo requerido')
    # R4: Criterio de edad
    if paciente['edad'] < servicio['edad_minima']:
        razones.append('R4: Paciente fuera del rango de edad para este protocolo')
    return razones

In [4]:
def calcular_puntuacion_salud(servicio, paciente):
    puntos = 0
    explicaciones = []
    # P1: Proximidad
    if servicio['zona'] == paciente['zona_residencia'] or servicio['zona'] == 'Todas':
        puntos += 20
        explicaciones.append('P1: Ubicación conveniente (+20)')
    # P2: Continuidad
    if servicio['centro'] == paciente['centro_previo']:
        puntos += 15
        explicaciones.append('P2: Continuidad de atención en mismo centro (+15)')
    # P3: Especialización por antecedentes
    if paciente['antecedentes_riesgo'] and servicio['especialidad'] == 'especialidad':
        puntos += 25
        explicaciones.append('P3: Especialista adecuado para antecedentes (+25)')
    # P4: Modalidad por severidad
    if paciente['dolor_intensidad'] >= 7 and servicio['tipo'] == 'presencial':
        puntos += 30
        explicaciones.append('P4: Atención presencial priorizada por dolor (+30)')
    return puntos, explicaciones

In [5]:
def recomendar_salud(paciente, servicios, n=3):
    recomendaciones = []
    descartadas = []
    for servicio in servicios:
        razones = evaluar_restricciones_salud(servicio, paciente)
        if razones:
            descartadas.append({'nombre': servicio['nombre'], 'razones': razones})
        else:
            puntos, explicaciones = calcular_puntuacion_salud(servicio, paciente)
            recomendaciones.append({'nombre': servicio['nombre'], 'puntos': puntos, 'explicaciones': explicaciones})

    recomendaciones.sort(key=lambda r: r['puntos'], reverse=True)
    return recomendaciones[:n], descartadas

In [6]:
from pprint import pprint

for paciente in [PACIENTE_1, PACIENTE_2]:
    recomendaciones, descartadas = recomendar_salud(paciente, SERVICIOS)
    print('\n===== Paciente: ' + paciente['nombre'] + ' =====')
    print('RECOMENDACIONES:')
    pprint(recomendaciones, sort_dicts=False)
    print('DESCARTADAS:')
    pprint(descartadas, sort_dicts=False)


===== Paciente: Pancracia =====
RECOMENDACIONES:
[{'nombre': 'Consulta medicina familiar - Centro A',
  'puntos': 65,
  'explicaciones': ['P1: Ubicación conveniente (+20)',
                    'P2: Continuidad de atención en mismo centro (+15)',
                    'P4: Atención presencial priorizada por dolor (+30)']},
 {'nombre': 'Urgencias ginecológicas - Hospital B',
  'puntos': 30,
  'explicaciones': ['P4: Atención presencial priorizada por dolor (+30)']},
 {'nombre': 'Orientación médica virtual',
  'puntos': 20,
  'explicaciones': ['P1: Ubicación conveniente (+20)']}]
DESCARTADAS:
[{'nombre': 'Ginecología - Hospital B',
  'razones': ['R1: Tiempo de espera supera la tolerancia clínica',
              'R2: Requiere pase a especialidad']},
 {'nombre': 'Mastografía preventiva - Clínica C',
  'razones': ['R1: Tiempo de espera supera la tolerancia clínica',
              'R4: Paciente fuera del rango de edad para este protocolo']}]

===== Paciente: Ricarda =====
RECOMENDACIONES:
[{'no

### Breve interpretación de las recomendaciones

**1. Paciente 1: Pancracia (Atención de primer nivel por síntoma agudo)**
- **Servicio priorizado:** Consulta Medicina Familiar - Centro A (65 puntos).
- **Servicios descartados:** Ginecología en Hospital B y Mastografía en Clínica C.
El recomendador actuó ante un cuadro de dolor agudo (8/10) que requería atención inmediata. Ya que no tenía una derivación médica previa y tenía tolerancia de espera de solo un día, las reglas R1 y R2 descartaron la cita directa con el especialista. El sistema priorizó la atención presencial de primer nivel en el centro de salud habitual de la paciente, para garantizar una valoración médica oportuna sin saturar los servicios de especialidad.

**2. Paciente 2: Ricarda (Atención especializada y diagnóstico)**
- **Servicios priorizados:** Ginecología en Hospital B (60 puntos) y Urgencias Ginecológicas en Hospital B (60 puntos).
- **Servicios descartados:** Medicina Familiar en Centro A y Orientación Virtual.
La paciente necesitaba un estudio que utiliza equipo técnico especializado y contaba con un pase previo y antecedentes de riesgo. La regla R3 eliminó las opciones de medicina familiar y orientación virtual por falta de capacidad técnica. Se canalizó a la paciente al hospital de segundo nivel donde se atiende habitualmente, se aprovechó el margen de tiempo de espera de 15 días para programar la atención especializada adecuada.

**Uso responsable de la IA:** Revisión de código con Gemini 3.7 Pro para detectar error al sustituir datos.